# HOL 1: Data Engineering Pipeline
## Raw → Silver → Gold Transformation with Snowpark Python

In this hands-on lab, you will build a data transformation pipeline using Snowpark Python.

**What you'll learn:**
- How to use Snowpark DataFrames to transform data
- Building a medallion architecture (Raw → Silver → Gold)
- Data quality profiling techniques
- Creating business-ready aggregated tables

**Prerequisites:** You must have run `setup.sql` first.

---

## Step 1: Connect to Snowflake and Verify Data
Run the cell below to establish your Snowpark session and verify the raw data is available.

In [ ]:
from snowflake.snowpark import Session
from snowflake.snowpark import functions as F
from snowflake.snowpark import types as T
from snowflake.snowpark.window import Window
from snowflake.snowpark.context import get_active_session

session = get_active_session()

session.use_database('BB_TRAINING')
session.use_schema('RAW')
session.use_warehouse('BB_TRAINING_WH')

print("Connected successfully!")
print(f"Database: {session.get_current_database()}")
print(f"Schema: {session.get_current_schema()}")
print(f"Warehouse: {session.get_current_warehouse()}")

In [ ]:
# Verify raw data exists
raw_customers = session.table('RAW.RAW_CUSTOMERS')
raw_applications = session.table('RAW.RAW_LOAN_APPLICATIONS')
raw_transactions = session.table('RAW.RAW_TRANSACTIONS')
raw_risk = session.table('RAW.RAW_RISK_ASSESSMENTS')

print("=== Raw Data Verification ===")
print(f"RAW_CUSTOMERS:         {raw_customers.count():,} rows")
print(f"RAW_LOAN_APPLICATIONS: {raw_applications.count():,} rows")
print(f"RAW_TRANSACTIONS:      {raw_transactions.count():,} rows")
print(f"RAW_RISK_ASSESSMENTS:  {raw_risk.count():,} rows")
print("\nAll tables verified! Ready to build the pipeline.")

---
## Step 2: Silver Layer - Clean & Validate

The Silver layer applies:
- **Deduplication** - Remove any duplicate records
- **Type casting** - Ensure correct data types
- **Null handling** - Filter or fill missing values
- **Standardization** - Consistent formatting
- **Enrichment** - Add calculated fields

We'll create 3 Silver tables:
1. `SILVER_CUSTOMERS` - Clean customer profiles
2. `SILVER_LOAN_APPLICATIONS` - Enriched with risk data
3. `SILVER_TRANSACTIONS` - Validated payment records

### 2a. SILVER_CUSTOMERS
Clean and standardize customer profiles.

In [ ]:
# Build SILVER_CUSTOMERS
# Transformations:
#   - Deduplicate on customer_id (keep most recent)
#   - Standardize state codes (uppercase, trim)
#   - Filter out records with invalid/null ABN
#   - Add revenue_band classification

silver_customers = (
    raw_customers
    # Remove duplicates - keep the most recent record
    .with_column('row_num', 
        F.row_number().over(
            Window.partition_by('CUSTOMER_ID').order_by(F.col('CREATED_AT').desc())
        )
    )
    .filter(F.col('row_num') == 1)
    .drop('row_num')
    
    # Filter out null ABNs
    .filter(F.col('ABN').is_not_null())
    .filter(F.length(F.col('ABN')) == 11)
    
    # Standardize state codes
    .with_column('STATE', F.upper(F.trim(F.col('STATE'))))
    
    # Add revenue band classification
    .with_column('REVENUE_BAND',
        F.when(F.col('ANNUAL_REVENUE') < 500000, F.lit('Micro'))
         .when(F.col('ANNUAL_REVENUE') < 2000000, F.lit('Small'))
         .when(F.col('ANNUAL_REVENUE') < 10000000, F.lit('Medium'))
         .otherwise(F.lit('Large'))
    )
    
    # Add employee band
    .with_column('EMPLOYEE_BAND',
        F.when(F.col('EMPLOYEES') < 5, F.lit('1-4'))
         .when(F.col('EMPLOYEES') < 20, F.lit('5-19'))
         .when(F.col('EMPLOYEES') < 100, F.lit('20-99'))
         .otherwise(F.lit('100+'))
    )
)

# Write to Silver schema
silver_customers.write.mode('overwrite').save_as_table('SILVER.SILVER_CUSTOMERS')

# Show results
result = session.table('SILVER.SILVER_CUSTOMERS')
print(f"SILVER_CUSTOMERS created: {result.count()} rows")
result.show(5)

### 2b. SILVER_LOAN_APPLICATIONS
Enrich loan applications with risk assessment data and add calculated fields.

In [ ]:
# Build SILVER_LOAN_APPLICATIONS
# Transformations:
#   - Join with risk assessments (1:1 on application_id)
#   - Add days_to_decision calculated field
#   - Filter out Withdrawn applications
#   - Add application_month for time-series analysis

silver_applications = (
    raw_applications
    # Filter out withdrawn - they're not useful for analysis
    .filter(F.col('STATUS') != 'Withdrawn')
    
    # Join with risk assessments
    .join(
        raw_risk.select(
            'APPLICATION_ID',
            'CREDIT_SCORE',
            'RISK_TIER',
            'DEBT_SERVICE_RATIO',
            'COLLATERAL_VALUE',
            'MODEL_VERSION'
        ),
        on='APPLICATION_ID',
        how='left'
    )
    
    # Add days_to_decision
    .with_column('DAYS_TO_DECISION',
        F.datediff('day', F.col('APPLICATION_DATE'), F.col('DECISION_DATE'))
    )
    
    # Add application_month for time-series grouping
    .with_column('APPLICATION_MONTH',
        F.date_trunc('month', F.col('APPLICATION_DATE'))
    )
    
    # Add approval flag for easier aggregation
    .with_column('IS_APPROVED',
        F.when(F.col('STATUS') == 'Approved', F.lit(1)).otherwise(F.lit(0))
    )
)

# Write to Silver schema
silver_applications.write.mode('overwrite').save_as_table('SILVER.SILVER_LOAN_APPLICATIONS')

# Show results
result = session.table('SILVER.SILVER_LOAN_APPLICATIONS')
print(f"SILVER_LOAN_APPLICATIONS created: {result.count()} rows")
result.select('APPLICATION_ID', 'LOAN_PRODUCT', 'LOAN_AMOUNT', 'STATUS', 
              'CREDIT_SCORE', 'RISK_TIER', 'DAYS_TO_DECISION').show(5)

### 2c. SILVER_TRANSACTIONS
Validate and clean transaction records.

In [ ]:
# Build SILVER_TRANSACTIONS
# Transformations:
#   - Remove Failed transactions (they didn't actually process)
#   - Add transaction_month for time-series
#   - Add is_late flag for late fee transactions
#   - Validate amounts (must be positive)

silver_transactions = (
    raw_transactions
    # Remove failed transactions
    .filter(F.col('STATUS') != 'Failed')
    
    # Ensure positive amounts
    .filter(F.col('AMOUNT') > 0)
    
    # Add is_late flag
    .with_column('IS_LATE_PAYMENT',
        F.when(F.col('TRANSACTION_TYPE') == 'Late Fee', F.lit(True))
         .otherwise(F.lit(False))
    )
    
    # Add transaction_month
    .with_column('TRANSACTION_MONTH',
        F.date_trunc('month', F.col('TRANSACTION_DATE'))
    )
    
    # Categorize transaction amounts
    .with_column('AMOUNT_CATEGORY',
        F.when(F.col('AMOUNT') < 1000, F.lit('Small'))
         .when(F.col('AMOUNT') < 10000, F.lit('Medium'))
         .when(F.col('AMOUNT') < 100000, F.lit('Large'))
         .otherwise(F.lit('Very Large'))
    )
)

# Write to Silver schema
silver_transactions.write.mode('overwrite').save_as_table('SILVER.SILVER_TRANSACTIONS')

# Show results
result = session.table('SILVER.SILVER_TRANSACTIONS')
print(f"SILVER_TRANSACTIONS created: {result.count()} rows")
result.show(5)

---
## Step 3: Data Quality Profiling

Before building the Gold layer, let's profile our Silver data to understand:
- How much data was filtered (raw vs silver counts)
- Null percentages by column
- Distribution of key dimensions

This is a critical step in any data engineering pipeline!

In [ ]:
# Data Quality Report: Row Count Comparison
print("=" * 60)
print("DATA QUALITY REPORT: Row Count Comparison")
print("=" * 60)
print(f"{'Table':<30} {'Raw':>8} {'Silver':>8} {'Filtered':>10}")
print("-" * 60)

raw_c = raw_customers.count()
silver_c = session.table('SILVER.SILVER_CUSTOMERS').count()
print(f"{'Customers':<30} {raw_c:>8,} {silver_c:>8,} {raw_c - silver_c:>10,}")

raw_a = raw_applications.count()
silver_a = session.table('SILVER.SILVER_LOAN_APPLICATIONS').count()
print(f"{'Loan Applications':<30} {raw_a:>8,} {silver_a:>8,} {raw_a - silver_a:>10,}")

raw_t = raw_transactions.count()
silver_t = session.table('SILVER.SILVER_TRANSACTIONS').count()
print(f"{'Transactions':<30} {raw_t:>8,} {silver_t:>8,} {raw_t - silver_t:>10,}")

print("\n** Records filtered due to: duplicates, invalid data,")
print("   withdrawn applications, and failed transactions.")

In [ ]:
# Data Quality Report: Key Dimension Distributions
print("\n" + "=" * 60)
print("DISTRIBUTION: Customers by Industry")
print("=" * 60)

industry_dist = (
    session.table('SILVER.SILVER_CUSTOMERS')
    .group_by('INDUSTRY')
    .agg(F.count('*').alias('COUNT'))
    .sort(F.col('COUNT').desc())
)
industry_dist.show()

print("\n" + "=" * 60)
print("DISTRIBUTION: Applications by Status & Avg Loan Amount")
print("=" * 60)

status_dist = (
    session.table('SILVER.SILVER_LOAN_APPLICATIONS')
    .group_by('STATUS')
    .agg(
        F.count('*').alias('COUNT'),
        F.round(F.avg('LOAN_AMOUNT'), 2).alias('AVG_LOAN_AMOUNT')
    )
    .sort(F.col('COUNT').desc())
)
status_dist.show()

print("\n" + "=" * 60)
print("DISTRIBUTION: Risk Tiers")
print("=" * 60)

risk_dist = (
    session.table('SILVER.SILVER_LOAN_APPLICATIONS')
    .group_by('RISK_TIER')
    .agg(
        F.count('*').alias('COUNT'),
        F.round(F.avg('CREDIT_SCORE'), 0).alias('AVG_CREDIT_SCORE')
    )
    .sort(F.col('AVG_CREDIT_SCORE').desc())
)
risk_dist.show()

---
## Step 4: Gold Layer - Business-Ready Aggregates

The Gold layer creates business-ready tables optimized for analytics:
1. **GOLD_LOAN_PERFORMANCE** - Per-loan metrics (repayment tracking)
2. **GOLD_CUSTOMER_360** - Complete customer view (all loans + risk)
3. **GOLD_PORTFOLIO_SUMMARY** - Monthly portfolio metrics (executive reporting)

### 4a. GOLD_LOAN_PERFORMANCE
Per-loan metrics showing repayment health.

In [ ]:
# Build GOLD_LOAN_PERFORMANCE
# For each approved loan: total repaid, outstanding balance, late payments

silver_apps = session.table('SILVER.SILVER_LOAN_APPLICATIONS')
silver_txns = session.table('SILVER.SILVER_TRANSACTIONS')

# Get approved loans with their transaction summaries
loan_txn_summary = (
    silver_txns
    .group_by('LOAN_ID')
    .agg(
        F.sum(
            F.when(F.col('TRANSACTION_TYPE') == 'Repayment', F.col('AMOUNT'))
             .otherwise(F.lit(0))
        ).alias('TOTAL_REPAID'),
        F.sum(
            F.when(F.col('TRANSACTION_TYPE') == 'Interest', F.col('AMOUNT'))
             .otherwise(F.lit(0))
        ).alias('TOTAL_INTEREST_PAID'),
        F.sum(
            F.when(F.col('TRANSACTION_TYPE') == 'Late Fee', F.col('AMOUNT'))
             .otherwise(F.lit(0))
        ).alias('TOTAL_LATE_FEES'),
        F.count(
            F.when(F.col('TRANSACTION_TYPE') == 'Late Fee', F.lit(1))
        ).alias('LATE_PAYMENT_COUNT'),
        F.count(
            F.when(F.col('TRANSACTION_TYPE') == 'Repayment', F.lit(1))
        ).alias('REPAYMENT_COUNT'),
        F.max('TRANSACTION_DATE').alias('LAST_PAYMENT_DATE')
    )
)

# Join with application details for approved loans
gold_loan_performance = (
    silver_apps
    .filter(F.col('STATUS') == 'Approved')
    .select(
        'APPLICATION_ID',
        'CUSTOMER_ID',
        'LOAN_PRODUCT',
        'LOAN_AMOUNT',
        'LOAN_TERM_MONTHS',
        'INTEREST_RATE',
        'RISK_TIER',
        'CREDIT_SCORE',
        'APPLICATION_DATE',
        'DECISION_DATE'
    )
    .join(
        loan_txn_summary,
        silver_apps['APPLICATION_ID'] == loan_txn_summary['LOAN_ID'],
        how='left'
    )
    .drop('LOAN_ID')
    
    # Calculate repayment ratio
    .with_column('REPAYMENT_RATIO',
        F.round(
            F.coalesce(F.col('TOTAL_REPAID'), F.lit(0)) / F.col('LOAN_AMOUNT') * 100, 2
        )
    )
    
    # Calculate payment health score (0-100)
    .with_column('PAYMENT_HEALTH_SCORE',
        F.greatest(F.lit(0),
            F.lit(100) 
            - (F.coalesce(F.col('LATE_PAYMENT_COUNT'), F.lit(0)) * F.lit(10))
            - F.when(F.coalesce(F.col('REPAYMENT_RATIO'), F.lit(0)) < 20, F.lit(20))
              .otherwise(F.lit(0))
        )
    )
)

# Write to Gold schema
gold_loan_performance.write.mode('overwrite').save_as_table('GOLD.GOLD_LOAN_PERFORMANCE')

result = session.table('GOLD.GOLD_LOAN_PERFORMANCE')
print(f"GOLD_LOAN_PERFORMANCE created: {result.count()} rows")
result.select('APPLICATION_ID', 'LOAN_PRODUCT', 'LOAN_AMOUNT', 
              'TOTAL_REPAID', 'REPAYMENT_RATIO', 'PAYMENT_HEALTH_SCORE').show(5)

### 4b. GOLD_CUSTOMER_360
Complete customer view aggregating all loan and payment activity.

In [ ]:
# Build GOLD_CUSTOMER_360
# One row per customer with all their loan portfolio metrics

silver_cust = session.table('SILVER.SILVER_CUSTOMERS')
gold_loans = session.table('GOLD.GOLD_LOAN_PERFORMANCE')

# Aggregate loan metrics per customer
customer_loan_metrics = (
    gold_loans
    .group_by('CUSTOMER_ID')
    .agg(
        F.count('*').alias('TOTAL_LOANS'),
        F.sum('LOAN_AMOUNT').alias('TOTAL_EXPOSURE'),
        F.sum('TOTAL_REPAID').alias('TOTAL_REPAID'),
        F.avg('CREDIT_SCORE').alias('AVG_CREDIT_SCORE'),
        F.avg('PAYMENT_HEALTH_SCORE').alias('AVG_PAYMENT_HEALTH'),
        F.sum('LATE_PAYMENT_COUNT').alias('TOTAL_LATE_PAYMENTS'),
        F.max('LAST_PAYMENT_DATE').alias('LAST_ACTIVITY_DATE'),
        F.avg('INTEREST_RATE').alias('AVG_INTEREST_RATE')
    )
)

# Join with customer details
gold_customer_360 = (
    silver_cust
    .select(
        'CUSTOMER_ID', 'BUSINESS_NAME', 'INDUSTRY', 'STATE',
        'ANNUAL_REVENUE', 'EMPLOYEES', 'YEARS_IN_BUSINESS',
        'REVENUE_BAND', 'EMPLOYEE_BAND'
    )
    .join(customer_loan_metrics, on='CUSTOMER_ID', how='left')
    
    # Fill nulls for customers with no loans
    .na.fill({
        'TOTAL_LOANS': 0,
        'TOTAL_EXPOSURE': 0,
        'TOTAL_REPAID': 0,
        'TOTAL_LATE_PAYMENTS': 0
    })
    
    # Add customer risk category
    .with_column('CUSTOMER_RISK_CATEGORY',
        F.when(F.col('AVG_PAYMENT_HEALTH') >= 80, F.lit('Low Risk'))
         .when(F.col('AVG_PAYMENT_HEALTH') >= 50, F.lit('Medium Risk'))
         .when(F.col('AVG_PAYMENT_HEALTH').is_not_null(), F.lit('High Risk'))
         .otherwise(F.lit('No Loan History'))
    )
)

# Write to Gold schema
gold_customer_360.write.mode('overwrite').save_as_table('GOLD.GOLD_CUSTOMER_360')

result = session.table('GOLD.GOLD_CUSTOMER_360')
print(f"GOLD_CUSTOMER_360 created: {result.count()} rows")
result.select('CUSTOMER_ID', 'BUSINESS_NAME', 'INDUSTRY', 
              'TOTAL_LOANS', 'TOTAL_EXPOSURE', 'AVG_PAYMENT_HEALTH',
              'CUSTOMER_RISK_CATEGORY').show(5)

### 4c. GOLD_PORTFOLIO_SUMMARY
Monthly portfolio metrics for executive reporting and trend analysis.

In [ ]:
# Build GOLD_PORTFOLIO_SUMMARY
# Monthly aggregates by industry and state for trend analysis

silver_apps = session.table('SILVER.SILVER_LOAN_APPLICATIONS')
silver_cust = session.table('SILVER.SILVER_CUSTOMERS')

# Join applications with customer info for dimensions
apps_with_customer = (
    silver_apps
    .join(
        silver_cust.select('CUSTOMER_ID', 'INDUSTRY', 'STATE', 'REVENUE_BAND'),
        on='CUSTOMER_ID',
        how='left'
    )
)

# Monthly portfolio summary
gold_portfolio_summary = (
    apps_with_customer
    .group_by('APPLICATION_MONTH', 'INDUSTRY', 'STATE', 'LOAN_PRODUCT')
    .agg(
        F.count('*').alias('TOTAL_APPLICATIONS'),
        F.sum('IS_APPROVED').alias('TOTAL_APPROVALS'),
        F.sum('LOAN_AMOUNT').alias('TOTAL_LOAN_AMOUNT'),
        F.avg('LOAN_AMOUNT').alias('AVG_LOAN_AMOUNT'),
        F.avg('CREDIT_SCORE').alias('AVG_CREDIT_SCORE'),
        F.avg('DAYS_TO_DECISION').alias('AVG_DAYS_TO_DECISION'),
        F.count_distinct('CUSTOMER_ID').alias('UNIQUE_CUSTOMERS')
    )
    
    # Add approval rate
    .with_column('APPROVAL_RATE',
        F.round(F.col('TOTAL_APPROVALS') / F.col('TOTAL_APPLICATIONS') * 100, 1)
    )
    
    # Sort for readability
    .sort(F.col('APPLICATION_MONTH').desc(), F.col('INDUSTRY'))
)

# Write to Gold schema
gold_portfolio_summary.write.mode('overwrite').save_as_table('GOLD.GOLD_PORTFOLIO_SUMMARY')

result = session.table('GOLD.GOLD_PORTFOLIO_SUMMARY')
print(f"GOLD_PORTFOLIO_SUMMARY created: {result.count()} rows")
result.show(10)

---
## Step 5: Final Verification

Let's verify our complete pipeline output.

In [ ]:
# Final Pipeline Summary
print("=" * 60)
print("PIPELINE COMPLETE: Final Table Summary")
print("=" * 60)
print(f"\n{'Layer':<10} {'Table':<35} {'Rows':>10}")
print("-" * 60)

tables = [
    ('RAW', 'RAW.RAW_CUSTOMERS'),
    ('RAW', 'RAW.RAW_LOAN_APPLICATIONS'),
    ('RAW', 'RAW.RAW_TRANSACTIONS'),
    ('RAW', 'RAW.RAW_RISK_ASSESSMENTS'),
    ('SILVER', 'SILVER.SILVER_CUSTOMERS'),
    ('SILVER', 'SILVER.SILVER_LOAN_APPLICATIONS'),
    ('SILVER', 'SILVER.SILVER_TRANSACTIONS'),
    ('GOLD', 'GOLD.GOLD_LOAN_PERFORMANCE'),
    ('GOLD', 'GOLD.GOLD_CUSTOMER_360'),
    ('GOLD', 'GOLD.GOLD_PORTFOLIO_SUMMARY'),
]

for layer, table in tables:
    count = session.table(table).count()
    print(f"{layer:<10} {table:<35} {count:>10,}")

print("\n" + "=" * 60)
print("Pipeline complete! Your Gold layer is ready for analytics.")
print("Proceed to the Streamlit dashboard section.")
print("=" * 60)

---
## Congratulations!

You've successfully built a complete data engineering pipeline:

- **Raw Layer** (4 tables) - Landing zone with synthetic business lending data
- **Silver Layer** (3 tables) - Cleaned, validated, and enriched
- **Gold Layer** (3 tables) - Business-ready aggregates for analytics

### Key Concepts Applied:
- Snowpark DataFrame transformations (filter, join, group_by, agg)
- Window functions for deduplication
- Data quality profiling
- Medallion architecture pattern
- Idempotent pipeline design (CREATE OR REPLACE via overwrite mode)

### Next: Build a Streamlit Dashboard
Continue to the Streamlit section to visualize your Gold layer data!